# control

> A local socket a detached job runner listens on, so a host's page can prompt, steer, stop and answer its agent from another process.

In [ ]:
#| default_exp control

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| hide
from fastcore.test import test_eq

In [ ]:
#| export
import json, os, socket, socketserver, stat, tempfile, threading
from dataclasses import asdict
from pathlib import Path
from ramabana.core import agent_err
from ramabana.agent import TurnResult

## The socket

A host runs each agent job in its own process, so the job survives a web-server restart. `Control` puts a Unix socket in the session's run folder, readable by this user only. Each request is one JSON line, `{"id", "method", "params"}`. Each reply is one JSON line, `{"id", "result"}` or `{"id", "error"}`.

The methods are `METHODS`:

- `prompt(text)` starts a turn on a worker thread, or queues `text` behind the turn that is running.
- `steer(text)` sends `text` to the running turn's next tool result.
- `cancel()` stops the running turn.
- `asks()` lists the asks waiting for an answer, oldest first. `answer(id, ok, note)` answers one.
- `status()` says whether a turn runs, how many prompts wait, and how the last turn ended.
- `events(offset)` reads the session's event file from byte `offset`, and returns the next offset. It needs `agent.events_to`.

`control_call(path, method, **params)` is the page's side: one request, its `result` back, and `RuntimeError` for an error.

In [ ]:
#| export
CONTROL = 'control.sock'   #: in a session's run folder: the socket `Control` listens on
METHODS = ('prompt', 'steer', 'cancel', 'asks', 'answer', 'status', 'events')
SOCKET_MAX = 100   #: bytes a Unix socket path may take; macOS allows 104
REQUEST_MAX = 1_000_000   #: bytes one request line may take
EVENTS_MAX = 1_000_000    #: bytes of the event file one `events` call returns

class Control:
    "A Unix socket for an agent run in a detached process: one JSON request a line, one JSON reply a line."
    def __init__(self, agent,
                 path=None):   # the socket; None puts it in the session's run folder as `CONTROL`
        self.agent, self.last = agent, None
        self.path = Path(path) if path else Path(agent.runs_dir)/CONTROL
        if not path and len(str(self.path).encode()) > SOCKET_MAX:   # a deep config folder: a short name in the temp folder
            self.path = Path(tempfile.gettempdir())/f'ramabana-{agent.session_id}.sock'
        if len(str(self.path).encode()) > SOCKET_MAX: raise ValueError(f'a Unix socket path is at most {SOCKET_MAX} bytes: {self.path}')
        self._lock, self._running, self._server, self._inode = threading.Lock(), False, None, None

    def start(self):
        "Listen on `path`, readable by this user only. Refuses a path another runner answers on, or one that is not a socket. Returns self."
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if self.path.exists() or self.path.is_symlink():
            if not stat.S_ISSOCK(self.path.lstat().st_mode): raise RuntimeError(f'{self.path} is not a socket; nothing was removed')
            with socket.socket(socket.AF_UNIX) as probe:
                try: probe.connect(str(self.path))
                except OSError: self.path.unlink()   # a runner that crashed left it
                else: raise RuntimeError(f'another runner listens on {self.path}')
        ctl = self
        class Handler(socketserver.StreamRequestHandler):
            def handle(self):
                while (raw := self.rfile.readline(REQUEST_MAX)):
                    try: req = json.loads(raw)
                    except ValueError as e: req = e
                    out = ctl.handle(req) if isinstance(req, dict) else {'id': None, 'error': {'message': f'not a JSON request object: {req}'}}
                    self.wfile.write((json.dumps(out, default=str) + '\n').encode())
                    self.wfile.flush()
        self._server = socketserver.ThreadingUnixStreamServer(str(self.path), Handler)
        os.chmod(self.path, 0o600)   # connecting needs write access; the default umask already keeps it from others
        self._inode = self.path.stat().st_ino
        self._server.daemon_threads = True
        threading.Thread(target=self._server.serve_forever, daemon=True, name='ramabana-control').start()
        return self

    def close(self):
        "Stop listening, and remove the socket while it is still this one's."
        if self._server is not None: self._server.shutdown(); self._server.server_close(); self._server = None
        try:
            if self._inode is not None and self.path.stat().st_ino == self._inode: self.path.unlink()
        except OSError: pass
        self._inode = None

    def handle(self, req):
        "One request as a dict; its reply as `{'id', 'result'}` or `{'id', 'error'}`."
        rid, name, params = req.get('id'), req.get('method', ''), req.get('params') or {}
        if name not in METHODS: return {'id': rid, 'error': {'message': f'no method {name!r}; one of {", ".join(METHODS)}'}}
        try: return {'id': rid, 'result': getattr(self, f'do_{name}')(**params)}
        except Exception as e: return {'id': rid, 'error': {'message': agent_err(e)}}

    def _work(self, text):
        "Run `text`, then each prompt queued meanwhile; under the lock, the last check and the end of running are one step."
        try:
            while text is not None:
                try: self.last = self.agent.ask(text, result=True)
                except Exception as e: self.last = TurnResult(state='failed', error=agent_err(e), session=self.agent.session_id)
                with self._lock:
                    text = self.agent.queued.popleft() if self.agent.queued else None
                    if text is None: self._running = False
        except BaseException:   # only an error that escaped: a normal end already cleared it, and a new turn may have set it since
            with self._lock: self._running = False
            raise

    def do_prompt(self, text):
        "Start a turn on `text`, or queue it behind the one running."
        with self._lock:
            if self._running: return {'queued': self.agent.queue(text)}
            self._running = True
        threading.Thread(target=self._work, args=(text,), daemon=True, name='ramabana-control-turn').start()
        return {'started': True}

    def do_steer(self, text): return {'told': self.agent.steer(text)}
    def do_cancel(self):
        "Stop the running turn, and drop the prompts queued behind it."
        with self._lock:
            dropped = len(self.agent.queued)
            self.agent.queued.clear()
        return self.agent.cancel() | {'dropped': dropped}
    def do_asks(self): return [a.dict() for a in self.agent.approvals.waiting] if self.agent.approvals is not None else []

    def do_answer(self, id, ok, note=''):
        ap = self.agent.approvals
        return ap is not None and ap.answer(id, bool(ok), note) is not None

    def do_status(self):
        return {'running': self._running, 'queued': len(self.agent.queued), 'session': self.agent.session_id,
                'last': asdict(self.last) if self.last is not None else None}

    def do_events(self, offset=0):
        files = self.agent.__dict__.get('_event_files') or ()
        if not files: raise RuntimeError('no event file; the runner calls agent.events_to(path) first')
        p = files[0][0]
        rotated = offset > p.stat().st_size   # the file rolled over to `.1` since the page last read it
        if rotated: offset = 0
        with open(p, 'rb') as f:
            f.seek(offset)
            data = f.read(EVENTS_MAX)
        cut = data.rfind(b'\n') + 1   # whole lines only: one still being written waits for the next read
        return {'events': [json.loads(l) for l in data[:cut].splitlines() if l.strip()], 'offset': offset + cut, 'rotated': rotated}


def control_call(path, method, timeout=30, **params):
    "Send one request to the `Control` socket at `path` and return its `result`; an error reply raises `RuntimeError`."
    with socket.socket(socket.AF_UNIX) as s:
        s.settimeout(timeout)
        s.connect(str(path))
        s.sendall((json.dumps({'id': 1, 'method': method, 'params': params}) + '\n').encode())
        out = json.loads(s.makefile('rb').readline())
    if 'error' in out: raise RuntimeError(out['error']['message'])
    return out['result']

A runner starts `Control` beside its agent, and a page talks to it through the socket. Here a scripted agent takes a prompt, the page reads the turn's events, and answers an ask raised while nobody else listens:

In [ ]:
import tempfile, time
from ramabana import agent
from ramabana.testing import fake_agent

d = Path(tempfile.mkdtemp(dir='/tmp'))   # a Unix socket path must stay short
a, be = fake_agent(cfg=d, replies=['the first reply'], approvals=agent.Approvals(tools={'edit_file'}, timeout=5, no_listener='wait'))
a.events_to(d/'job.events')
ctl = Control(a).start()
call = lambda method, **kw: control_call(ctl.path, method, **kw)

test_eq(call('prompt', text='hello'), {'started': True})
for _ in range(200):
    if not call('status')['running']: break
    time.sleep(.01)
st = call('status')
test_eq((st['running'], st['last']['state'], st['last']['reply']), (False, 'complete', 'the first reply'))
ev = call('events')
assert {'turn_start', 'turn_end'} <= {e['kind'] for e in ev['events']} and ev['offset'] > 0
test_eq(call('events', offset=ev['offset'])['events'], [])

In [ ]:
out = {}
threading.Thread(target=lambda: out.setdefault('r', a.approvals.request('edit_file', {'path': 'a.py'})), daemon=True).start()
for _ in range(200):
    if (waiting := call('asks')): break
    time.sleep(.01)
test_eq(waiting[0]['tool'], 'edit_file')
assert call('answer', id=waiting[0]['id'], ok=True)
for _ in range(200):
    if 'r' in out: break
    time.sleep(.01)
assert out['r'], 'answered through the socket'
try: call('fly')
except RuntimeError as e: assert 'no method' in str(e)
ctl.close(); a.close()

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()